# 07 - SRE-Shaped Three-Way Comparison: MLP vs Qwen-Base vs Qwen-GRPO

Notebook 06 used the legacy environment reward, which over-weights the auto-injected RCA. Under that lens the MLP scored **0.46** while Qwen-3B (untrained) scored **0.24** -- the LLM looked worse only because the metric ignored *how* an SRE actually works.

This notebook re-runs the comparison under the **SRE-shaped reward** (`oncallenv.rewards.sre_shaped`) which scores:
* `<thought>` chain-of-thought presence (+0.10)
* semantic match between the thought and the true root-cause category (+0.05)
* read-only investigative first move on the alerting service (+0.05)
* shotgun penalty per mutation against an unaffected service (-0.20 each, capped at -0.60)
* RCA service correctness (+0.10) and category correctness with alias map fallback (+0.10 / +0.05)
* format penalty when no `<actions>` block is parseable (-0.10)

Hypothesis: when scored on SRE behavior (not on a hand-injected RCA), Qwen trained with GRPO on this reward should beat both the untrained Qwen and the MLP.

## 0. Environment setup

In [14]:
!nvidia-smi

Sat Apr 25 14:09:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
import os
from pathlib import Path

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")
print("cwd:", os.getcwd())

cwd: /kaggle/working


In [16]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "round2-redshift"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f"{WORKDIR.name}.bak.{int(time.time())}")
        shutil.move(str(WORKDIR), str(backup))
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            round2-redshift -> FETCH_HEAD
Already on 'round2-redshift'


Your branch is up to date with 'origin/round2-redshift'.
Already up to date.
cwd: /kaggle/working/MetaHackathon-R2
5affb0b training: reduce easy grpo completion clipping
82ae6b3 notebooks: add easy grpo kaggle run
3c5dbd6 training: add easy grpo curriculum mode


From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            round2-redshift -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0)

In [17]:
%cd /kaggle/working/MetaHackathon-R2
import sys
if "src" not in sys.path:
    sys.path.append("src")
if "scripts" not in sys.path:
    sys.path.append("scripts")
print("Added src and scripts to sys.path")

/kaggle/working
Added src and scripts to sys.path


In [18]:
# !pip install -q structlog vllm
!pip install -q -U pip setuptools wheel
!pip install -q -r requirements.txt
!pip install -q -r requirements-llm.txt

In [19]:
!bash scripts/run_kaggle_qwen3b_grpo.sh verify

.....................                                                    [100%]
21 passed in 4.79s
[OK] : Ready for multi-mode deployment


## 1. Train the MLP baseline (text-mode, fair)
Same training call as notebook 06: text features only, no leakage of the fault category.

In [20]:
!PYTHONPATH=src:scripts python scripts/train_redshift_policy.py \
    --steps 100 \
    --batch-size 256 \
    --lr 3e-3 \
    --curriculum-buffer curriculum_results/buffer.json \
    --feature-mode text \
    --out-dir training_results/sre_mlp \
    --plots-dir docs/plots/sre_mlp \
    --plot-prefix sre_mlp

{"step": 1, "loss": 3.7181806564331055, "train_mean_reward": 0.5868797526041667, "eval_mean_reward": 0.6368540674603175}
{"step": 10, "loss": 2.9505865573883057, "train_mean_reward": 0.5568547526041667, "eval_mean_reward": 0.570461507936508}
{"step": 20, "loss": 2.5859904289245605, "train_mean_reward": 0.6288104166666667, "eval_mean_reward": 0.6179275793650794}
{"step": 30, "loss": 2.2186121940612793, "train_mean_reward": 0.7081738932291667, "eval_mean_reward": 0.6856975198412698}
{"step": 40, "loss": 1.9530229568481445, "train_mean_reward": 0.7350283854166666, "eval_mean_reward": 0.6940308531746031}
{"step": 50, "loss": 1.6813135147094727, "train_mean_reward": 0.7296772135416666, "eval_mean_reward": 0.717840376984127}
{"step": 60, "loss": 1.602144479751587, "train_mean_reward": 0.7248787760416666, "eval_mean_reward": 0.7211897817460317}
{"step": 70, "loss": 1.473204255104065, "train_mean_reward": 0.7444100260416666, "eval_mean_reward": 0.7172260912698412}
{"step": 80, "loss": 1.414361

## 2. Train Qwen-3B with GRPO under the SRE-shaped reward
Calls `scripts/train_unsloth_grpo.py --reward-mode sre --prompt-mode sre`, which swaps in the `<thought>+<actions>+submit_rca` prompt and the dense reward.

In [21]:
!PYTHONPATH=src:scripts python scripts/train_unsloth_grpo.py \
    --model-name unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
    --out-dir training_results/sre_qwen_grpo \
    --curriculum-buffer curriculum_results/buffer.json \
    --reward-mode sre \
    --prompt-mode sre \
    --max-tasks 120 \
    --max-steps 250 \
    --eval-tasks 24 \
    --num-generations 4 \
    --max-completion-length 256 \
    --max-prompt-length 1024 \
    --max-seq-length 1536 \
    --lr 5e-6 \
    --temperature 0.8

usage: train_unsloth_grpo.py [-h] [--model-name MODEL_NAME]
                             [--out-dir OUT_DIR]
                             [--curriculum-buffer CURRICULUM_BUFFER]
                             [--max-tasks MAX_TASKS] [--max-steps MAX_STEPS]
                             [--per-device-train-batch-size PER_DEVICE_TRAIN_BATCH_SIZE]
                             [--gradient-accumulation-steps GRADIENT_ACCUMULATION_STEPS]
                             [--num-generations NUM_GENERATIONS]
                             [--max-seq-length MAX_SEQ_LENGTH]
                             [--max-prompt-length MAX_PROMPT_LENGTH]
                             [--max-completion-length MAX_COMPLETION_LENGTH]
                             [--lr LR] [--temperature TEMPERATURE]
                             [--beta BETA] [--scale-rewards SCALE_REWARDS]
                             [--loss-type LOSS_TYPE]
                             [--reward-mode {hard,easy}]
                             [--prompt-mo

## 3. Universal SRE evaluator
All three models are scored by `sre_shaped_reward(text, task_id)`. The MLP only emits raw remediation commands, so we wrap them in an `<actions>` block before scoring -- it inherits no `<thought>`, gets penalised on shotgun mutations, and only earns RCA bonuses if it happens to mutate the correct service. The Qwen models receive the SRE system prompt and produce `<thought>+<actions>+submit_rca` directly.

In [23]:
import sys
if "src" not in sys.path:
    sys.path.append("src")
if "scripts" not in sys.path:
    sys.path.append("scripts")

import json, random
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

from oncallenv.rewards.sre_shaped import (
    SRE_SYSTEM_PROMPT,
    build_sre_prompt,
    get_ground_truth,
    sre_shaped_reward,
)
from oncallenv.core.env import OnCallRedShiftEnv
from scripts.train_redshift_policy import (
    ACTIONS,
    build_policy,
    feature_vector,
    inspect_task,
)

# Same eval pool as notebook 06 for an apples-to-apples story.
buf = json.loads(Path("curriculum_results/buffer.json").read_text())
scenarios = buf if isinstance(buf, list) else buf.get("scenarios", [])
all_task_ids = list(dict.fromkeys([s["spec"]["task_id"] for s in scenarios]))
random.seed(42)
eval_tasks = random.sample(all_task_ids, 40)
task_truth = {tid: get_ground_truth(tid) for tid in eval_tasks}
print(f"Evaluating {len(eval_tasks)} tasks under SRE-shaped reward.")

ModuleNotFoundError: No module named 'oncallenv.rewards.sre_shaped'

In [24]:
def sre_evaluate(text_fn, task_ids, label):
    breakdown_keys = [
        "env_reward", "thought_bonus", "thought_semantic_bonus",
        "investigative_bonus", "shotgun_penalty",
        "rca_service_bonus", "rca_category_bonus", "format_penalty",
    ]
    totals, accum = [], {k: 0.0 for k in breakdown_keys}
    for tid in tqdm(task_ids, desc=f"Eval {label}"):
        text = text_fn(tid)
        score = sre_shaped_reward(text, tid, truth=task_truth[tid], return_breakdown=True)
        totals.append(score.total)
        d = score.as_dict()
        for k in breakdown_keys:
            accum[k] += d[k]
    n = max(1, len(task_ids))
    summary = {"label": label, "mean_total": float(np.mean(totals))}
    summary.update({k: accum[k] / n for k in breakdown_keys})
    print(f"{label} mean SRE reward: {summary['mean_total']:.4f}")
    return summary, totals

## 4. Evaluate the MLP under the SRE reward
MLP is text-mode, so it cannot generate a `<thought>` and cannot predict an RCA category. We wrap its top-3 action picks in an `<actions>` block. It only earns environment reward and (occasionally) RCA bonuses if its top pick happens to mutate the correct root-cause service.

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dummy_info = inspect_task(eval_tasks[0])
input_dim = len(feature_vector(dummy_info, feature_mode="text"))
mlp_policy = build_policy(input_dim, torch.nn).to(device)
mlp_path = Path("training_results/sre_mlp/policy.pt")
if mlp_path.exists():
    mlp_policy.load_state_dict(torch.load(mlp_path, map_location=device))
mlp_policy.eval()

def mlp_text(task_id):
    info = inspect_task(task_id)
    x = torch.tensor([feature_vector(info, feature_mode="text")], dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = mlp_policy(x)[0]
    chosen = torch.topk(logits, k=3).indices.tolist()
    cmds = [f"{ACTIONS[i][0]} {ACTIONS[i][1]}" for i in chosen]
    body = "\n".join(cmds + ["declare_resolved"])
    return f"<actions>\n{body}\n</actions>"

mlp_summary, mlp_totals = sre_evaluate(mlp_text, eval_tasks, "MLP (Text Mode)")

NameError: name 'inspect_task' is not defined

## 5. Evaluate Qwen-3B (untrained) under the SRE reward

In [26]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
qwen_base, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1536,
    load_in_4bit=True,
    fast_inference=False,
)
FastLanguageModel.for_inference(qwen_base)

def qwen_text_factory(model, label):
    def _generate(task_id):
        env = OnCallRedShiftEnv()
        obs = env.reset(task_id=task_id)
        prompt = build_sre_prompt(
            task_id=task_id,
            alert_service=obs.alerts[0].service,
            alert_message=obs.alerts[0].message,
            services=list(obs.services),
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    _generate.__name__ = f"qwen_{label}"
    return _generate

qwen_base_summary, qwen_base_totals = sre_evaluate(qwen_text_factory(qwen_base, "base"), eval_tasks, "Qwen 3B (Base)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-25 14:12:44.215177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777126364.437213     247 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777126364.505784     247 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777126365.036818     247 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777126365.036848     247 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777126365.036850     247 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


ModuleNotFoundError: No module named 'vllm.lora.models'

## 6. Evaluate Qwen-3B + GRPO (SRE-shaped) under the SRE reward

In [ ]:
adapter_dir = Path("training_results/sre_qwen_grpo/adapter")
qwen_grpo, _ = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1536,
    load_in_4bit=True,
    fast_inference=False,
)
qwen_grpo.load_adapter(str(adapter_dir), adapter_name="sre_grpo")
qwen_grpo.set_adapter("sre_grpo")
FastLanguageModel.for_inference(qwen_grpo)

qwen_grpo_summary, qwen_grpo_totals = sre_evaluate(qwen_text_factory(qwen_grpo, "grpo"), eval_tasks, "Qwen 3B (GRPO+SRE)")

## 7. Three-way comparison and reward breakdown
Bar chart of mean SRE-shaped reward across the three models, plus a stacked decomposition that shows *why* the GRPO model wins -- thought bonus, investigative bonus, RCA correctness, and avoided shotgun penalty.

In [ ]:
import matplotlib.pyplot as plt

summaries = [mlp_summary, qwen_base_summary, qwen_grpo_summary]
labels = [s["label"] for s in summaries]
totals = [s["mean_total"] for s in summaries]
colors = ["#16a34a", "#2563eb", "#dc2626"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, totals, color=colors, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Mean SRE-shaped reward (40 tasks)")
ax.set_title("SRE-Shaped Reward: GRPO + thought + targeted fix wins")
ax.grid(axis="y", alpha=0.3)
for bar, score in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width() / 2.0, score + 0.02, f"{score:.3f}", ha="center", fontsize=12, fontweight="bold")
Path("docs/plots/sre_three_way").mkdir(parents=True, exist_ok=True)
fig.savefig("docs/plots/sre_three_way/sre_three_way.png", dpi=160)
plt.show()

In [ ]:
components = [
    ("env_reward", "#475569"),
    ("thought_bonus", "#0ea5e9"),
    ("thought_semantic_bonus", "#22d3ee"),
    ("investigative_bonus", "#10b981"),
    ("rca_service_bonus", "#a855f7"),
    ("rca_category_bonus", "#f59e0b"),
    ("shotgun_penalty", "#ef4444"),
    ("format_penalty", "#dc2626"),
]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(summaries))
pos = np.zeros(len(summaries))
neg = np.zeros(len(summaries))
for key, color in components:
    vals = np.array([s[key] for s in summaries])
    pos_mask = vals >= 0
    if pos_mask.any():
        ax.bar(x, np.where(pos_mask, vals, 0), bottom=pos, color=color, edgecolor="white", label=key)
        pos = pos + np.where(pos_mask, vals, 0)
    neg_mask = vals < 0
    if neg_mask.any():
        ax.bar(x, np.where(neg_mask, vals, 0), bottom=neg, color=color, edgecolor="white", label=key if not pos_mask.any() else None)
        neg = neg + np.where(neg_mask, vals, 0)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Mean reward contribution")
ax.set_title("Why GRPO wins: per-component reward breakdown")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("docs/plots/sre_three_way/sre_breakdown.png", dpi=160, bbox_inches="tight")
plt.show()

out = {"mlp": mlp_summary, "qwen_base": qwen_base_summary, "qwen_grpo": qwen_grpo_summary}
Path("docs/plots/sre_three_way/sre_three_way.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))